# 02. DATA CLEANING & OUTLIER TREATMENT
**Xử lý ngoại lệ bằng phương pháp IQR Capping và phân tích giá trị Unknown**
---
Nội dung notebook này thực hiện:
1. Thống kê mô tả xu hướng trung tâm và độ biến thiên (Mean, Median, Std, IQR).
2. Phát hiện giá trị ngoại lai (Outliers) trên các biến phân phối lệch (`balance`, `duration`, `campaign`).
3. Áp dụng kỹ thuật Capping (Winsorizing) theo ngưỡng $[Q_1 - 1.5\times IQR, Q_3 + 1.5\times IQR]$.
4. Lưu trữ tập dữ liệu sạch `bank_cleaned.csv`.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../dataset/Bank_Marketing_Dataset.csv')
column_mapping = {
    'V1': 'age', 'V2': 'job', 'V3': 'marital', 'V4': 'education',
    'V5': 'default', 'V6': 'balance', 'V7': 'housing', 'V8': 'loan',
    'V9': 'contact', 'V10': 'day', 'V11': 'month', 'V12': 'duration',
    'V13': 'campaign', 'V14': 'pdays', 'V15': 'previous', 'V16': 'poutcome',
    'Class': 'deposit'
}
df = df.rename(columns=column_mapping)
df['deposit'] = df['deposit'].map({1: 'no', 2: 'yes'})
df['target'] = (df['deposit'] == 'yes').astype(int)


In [ ]:
# Thống kê mô tả các biến định lượng
numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
stats_df = df[numeric_cols].describe().T
stats_df['IQR'] = df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25)
stats_df


In [ ]:
# Xử lý ngoại lệ bằng IQR Capping
df_clean = df.copy()
for col in ['balance', 'duration', 'campaign']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lb, ub = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = ((df[col] < lb) | (df[col] > ub)).sum()
    print(f'{col}: Giới hạn [{lb:.1f}, {ub:.1f}] -> Số ngoại lai: {outliers} ({outliers/len(df)*100:.2f}%)')
    df_clean[col] = df_clean[col].clip(lower=lb, upper=ub)

df_clean.to_csv('../dataset/cleaned/bank_cleaned.csv', index=False)
print('Đã lưu dữ liệu làm sạch tại dataset/cleaned/bank_cleaned.csv')
